# YOLO Object Detection with OpenCV

> **Advanced · Deep learning inference**


## Why this matters

YOLO is an excellent end-to-end case study: resize and normalize an input, decode a model-specific tensor, suppress duplicates, then measure the result on images and video.

**Where it appears:** Object detection prototypes, visual inspection tools, video analytics experiments, and downstream tracking input.


## Learning Objectives

- Understand YOLO architecture
- Parse YOLOv8 outputs


## Prerequisites

22 OpenCV DNN and ONNX Inference; 13 Video Processing and Background Motion for video sections

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

YOLOv8 ONNX, `blobFromImage`, box decoding, confidence filtering, `cv2.dnn.NMSBoxes`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### YOLO Object Detection with OpenCV

YOLO ('You Only Look Once') predicts bounding boxes and class probabilities
in a single forward pass across a grid of anchor positions. Modern architectures like **YOLOv8** have removed the "objectness" score and simplified the output format.

YOLOv8 outputs a tensor of shape `(1, 84, N)` (where 84 = 4 box coordinates + 80 classes, and N is the number of anchors). We transpose this to `(1, N, 84)` to easily extract the center, width, height, and class scores for each detection.

Raw YOLO output contains many overlapping, low-confidence candidate boxes per object; a full pipeline requires confidence filtering followed by **class-aware** non-max suppression (`cv2.dnn.NMSBoxes`) to reduce this to one clean box per real object.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 2. Confidence filtering and box decoding

Extract the maximum class score for each anchor to get the final confidence, and decode the normalized center/width/height into pixel-space corner coordinates. Note that YOLOv8 outputs are relative to the 640x640 input grid.


In [ ]:
def decode_yolo_output(
    raw_output: np.ndarray, image_shape: tuple, conf_threshold: float = 0.4
) -> list[dict]:
    h, w = image_shape[:2]
    # YOLOv8 scales boxes to the 640x640 input, so we compute scaling factors back to the original image
    x_factor = w / 640.0
    y_factor = h / 640.0
    
    detections = []
    # raw_output is (1, 8400, 84)
    for row in raw_output[0]:
        class_scores = row[4:]
        class_id = int(np.argmax(class_scores))
        confidence = float(class_scores[class_id])
        
        if confidence < conf_threshold:
            continue
            
        cx, cy, bw, bh = row[:4]
        x1 = int((cx - bw / 2) * x_factor)
        y1 = int((cy - bh / 2) * y_factor)
        box_w = int(bw * x_factor)
        box_h = int(bh * y_factor)
        
        detections.append(
            {
                "class_id": class_id,
                "class_name": CLASS_NAMES[class_id],
                "confidence": confidence,
                "box": (x1, y1, box_w, box_h),
            }
        )
    return detections


image_shape = img.shape
decoded = decode_yolo_output(raw_output, image_shape)
print(f"Found {len(decoded)} candidates before NMS")

# Draw to verify before NMS
img_raw_boxes = img.copy()
for d in decoded:
    x, y, w_box, h_box = d["box"]
    cv2.rectangle(img_raw_boxes, (x, y), (x + w_box, y + h_box), (0, 0, 255), 1)
show_grid([("All Candidate Boxes (Before NMS)", img_raw_boxes)])


### 3. Class-aware non-max suppression


In [ ]:
def apply_nms(
    detections: list[dict], nms_threshold: float = 0.4, conf_threshold: float = 0.4
) -> list[dict]:
    if not detections:
        return []
    
    boxes = [d["box"] for d in detections]
    scores = [d["confidence"] for d in detections]
    class_ids = [d["class_id"] for d in detections]
    
    indices = cv2.dnn.NMSBoxes(boxes, scores, conf_threshold, nms_threshold)
    if len(indices) == 0:
        return []
        
    return [detections[i] for i in indices.flatten()]

final_detections = apply_nms(decoded)
print(f"Found {len(final_detections)} objects after NMS")
img_final = img.copy()
for d in final_detections:
    x, y, w_box, h_box = d["box"]
    cv2.rectangle(img_final, (x, y), (x + w_box, y + h_box), (0, 255, 0), 2)
    cv2.putText(img_final, f"{d['class_name']} {d['confidence']:.2f}", (x, max(y - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
show_grid([("Final NMS Result", img_final)])


### 4. The complete inference function


In [ ]:
def yolo_detect_pipeline(
    raw_output: np.ndarray,
    image_shape: tuple,
    conf_threshold: float = 0.4,
    nms_threshold: float = 0.4,
) -> list[dict]:
    decoded = decode_yolo_output(raw_output, image_shape, conf_threshold)
    return apply_nms(decoded, nms_threshold, conf_threshold)

result = yolo_detect_pipeline(raw_output, img.shape)
print(f"Pipeline produced {len(result)} final detection(s)")


### 5. Processing Video
Let's put the full `yolo_detect_pipeline` to the test by running it on the `traffic.mp4` video.


In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(get_real_data("video", "traffic.mp4")))
frame_idx = 0

print("Processing traffic video... this might take a moment.")
while cap.isOpened() and frame_idx < 30:
    ret, frame = cap.read()
    if not ret:
        break
    
    blob = cv2.dnn.blobFromImage(frame, 1 / 255.0, (640, 640), swapRB=True, crop=False)
    net.setInput(blob)
    raw_output = net.forward()
    raw_output = np.transpose(raw_output, (0, 2, 1))
    
    result = yolo_detect_pipeline(raw_output, frame.shape)
    
    for d in result:
        x, y, w_box, h_box = d["box"]
        cv2.rectangle(frame, (x, y), (x + w_box, y + h_box), (0, 255, 0), 2)
        cv2.putText(frame, f"{d['class_name']} {d['confidence']:.2f}", (x, max(y - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(frame_rgb)
    plt.axis('off')
    plt.show()
    clear_output(wait=True)
    frame_idx += 1

cap.release()
print("Done processing 30 frames!")


### 3. Visualization

Now let's draw the filtered bounding boxes on the original image and plot the result.

In [ ]:
boxes = [d["box"] for d in decoded]
scores = [d["confidence"] for d in decoded]
class_ids = [d["class_id"] for d in decoded]
indices = cv2.dnn.NMSBoxes(boxes, scores, 0.4, 0.4)
if len(indices) > 0:
    indices = indices.flatten()

import matplotlib.pyplot as plt

img_display = img.copy()
for i in indices:
    box = boxes[i]
    score = scores[i]
    class_id = class_ids[i]
    
    x, y, w, h = box
    label = f"{CLASS_NAMES[class_id]}: {score:.2f}"
    
    cv2.rectangle(img_display, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(img_display, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('YOLOv8 Object Detection Results')
plt.show()


### Multiple Plots
Let's also look at cropped versions of each detected object to get a closer view.

In [ ]:
fig, axes = plt.subplots(1, len(indices), figsize=(15, 5))
if len(indices) == 1:
    axes = [axes]

for idx, i in enumerate(indices):
    x, y, w, h = boxes[i]
    # Ensure coordinates are within image boundaries
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(img.shape[1], x + w), min(img.shape[0], y + h)
    
    crop = img[y1:y2, x1:x2]
    label = f"{CLASS_NAMES[class_ids[i]]}"
    
    axes[idx].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(label)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()


## Summary

You can run a YOLO ONNX model through OpenCV, parse its outputs deliberately, apply class-aware NMS, and visualize valid detections.

- **Best Practices:** Pin the exact model/export version, keep labels paired with weights, inspect raw tensor shapes, and assess false positives/negatives on representative images.
- **Common Pitfalls:** Applying a YOLOv8 parser to another version, mixing letterbox and resize coordinates, and treating confidence as a calibrated probability.